In [ ]:
import numpy as np, os, time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def langevin_L(x):
    """Функция Ланжевена L(x) = coth(x) - 1/x с рядом Тейлора при малых x.

    Безопасна во всей области определения, в отличие от прямой аналитической
    формулы производной dMan/dHeff без regularization (расходится как
    1/Heff^2 при Heff->0 — см. проверку численно: при Heff=1e-8 наивная
    формула даёт ~3e15 вместо верного предела Ms/(3a) [2].
    """
    x = np.asarray(x, dtype=float)
    small = np.abs(x) < 1e-6
    return np.where(small, x/3.0 - x**3/45.0,
                     1.0/np.tanh(np.where(small, 1.0, x)) - 1.0/np.where(small, 1.0, x))


class JASmoothRK4Robust:
    """Модель гистерезиса Джилса-Атертона (СГС).

    Особенности реализации:
    - Подразбиение шага (n_sub) с интегратором RK4 для устойчивости при
      резких изменениях поля.
    - Итерация неподвижной точки (n_fp) для самосогласованного вычисления
      M через эффективное поле He = H + H0_bias + alpha*M.
    - Производная dM/dH вычисляется ЧИСЛЕННО (конечная разность), а не по
      аналитической формуле — избегаем сингулярности вблизи Heff=0.
    - Утечка магнитного поля (sigma_m_leak) для варьирования магнитных потерь.
    """
    def __init__(self, Ms, a, alpha, k, c, sigma_m_leak=0.0, H0_bias=0.0, n_sub=4, n_fp=3):
        self.Ms, self.a, self.alpha, self.k, self.c = Ms, a, alpha, k, c
        self.sigma_m_leak = sigma_m_leak
        self.H0_bias = H0_bias
        self.n_sub, self.n_fp = n_sub, n_fp

    def _M_of_Mirr(self, M_irr, H, M_guess):
        M = M_guess
        for _ in range(self.n_fp):
            He = (H + self.H0_bias) + self.alpha * M
            Man = self.Ms * langevin_L(He / self.a)
            M = (1 - self.c) * M_irr + self.c * Man
        He = (H + self.H0_bias) + self.alpha * M
        Man = self.Ms * langevin_L(He / self.a)
        return M, Man

    def _rhs(self, M_irr, H, delta, M_guess):
        M, Man = self._M_of_Mirr(M_irr, H, M_guess)
        denom = self.k * delta - self.alpha * (Man - M_irr)
        denom_safe = np.where(np.abs(denom) < 1e-9, 1e-9 * np.sign(denom + 1e-30), denom)
        return (Man - M_irr) / denom_safe, M

    def integrate_rk4(self, M_irr_old, H_old, H_new, M_guess_old, delta):
        h = (H_new - H_old) / self.n_sub
        M_irr, M_guess, H_cur = M_irr_old.copy(), M_guess_old.copy(), H_old.copy()
        for _ in range(self.n_sub):
            k1, M1 = self._rhs(M_irr, H_cur, delta, M_guess)
            k2, M2 = self._rhs(M_irr + 0.5*h*k1, H_cur + 0.5*h, delta, M1)
            k3, M3 = self._rhs(M_irr + 0.5*h*k2, H_cur + 0.5*h, delta, M2)
            k4, M4 = self._rhs(M_irr + h*k3, H_cur + h, delta, M3)
            M_irr = np.clip(M_irr + (h/6.0)*(k1 + 2*k2 + 2*k3 + k4), -self.Ms, self.Ms)
            H_cur = H_cur + h
            M_guess = M4
        M_final, _ = self._M_of_Mirr(M_irr, H_new, M_guess)
        return M_irr, np.clip(M_final, -self.Ms, self.Ms)

    def dM_dH(self, M_irr_old, H_old, H_new, M_guess_old, delta):
        eps = 1e-6 * np.maximum(1.0, np.abs(H_new))
        _, M0 = self.integrate_rk4(M_irr_old, H_old, H_new, M_guess_old, delta)
        _, M1 = self.integrate_rk4(M_irr_old, H_old, H_new + eps, M_guess_old, delta)
        return (M1 - M0) / eps, M0

    def _newton_branch(self, M_irr_old, H_old, M_old, rhs_extra, dt, delta, n_newton, max_step):
        H_new = H_old - rhs_extra
        leak_coef = 4*np.pi*self.sigma_m_leak*dt
        for _ in range(n_newton):
            dMdH, Mc = self.dM_dH(M_irr_old, H_old, H_new, M_old, delta)
            F = H_new - H_old + 4*np.pi*(Mc - M_old) + leak_coef*H_new + rhs_extra
            Fp = 1.0 + 4*np.pi*dMdH + leak_coef
            Fp_safe = np.where(np.abs(Fp) < 0.5, 0.5*np.sign(Fp + 1e-30), Fp)
            H_new = H_new - np.clip(F/Fp_safe, -max_step, max_step)
        dMdH, Mc = self.dM_dH(M_irr_old, H_old, H_new, M_old, delta)
        F_final = H_new - H_old + 4*np.pi*(Mc - M_old) + leak_coef*H_new + rhs_extra
        return H_new, F_final

    def implicit_step(self, M_irr_old, H_old, M_old, rhs_extra, dt,
                       n_newton=6, max_step_factor=0.25, tol=1e-5):
        """Неявный шаг по методу Ньютона с двусторонней проверкой знака delta=sign(dH/dt).

        Знак delta фиксируется единожды по знаку внешнего толчка (не
        пересчитывается внутри итераций Ньютона) — иначе возникает
        паразитный цикл период-2.
        """
        max_step = max_step_factor * self.a
        delta_pred = np.where(-rhs_extra >= 0, 1.0, -1.0)
        H_a, F_a = self._newton_branch(M_irr_old, H_old, M_old, rhs_extra, dt, delta_pred, n_newton, max_step)
        bad = np.abs(F_a) > tol
        H_new, delta_final = H_a.copy(), delta_pred.copy()
        if np.any(bad):
            delta_alt = -delta_pred
            H_b, F_b = self._newton_branch(M_irr_old, H_old, M_old, rhs_extra, dt, delta_alt, n_newton, max_step)
            use_b = bad & (np.abs(F_b) < np.abs(F_a))
            H_new = np.where(use_b, H_b, H_a)
            delta_final = np.where(use_b, delta_alt, delta_pred)
        M_irr_f, M_f = self.integrate_rk4(M_irr_old, H_old, H_new, M_old, delta_final)
        return H_new, M_irr_f, M_f


class LLGVector:
    """Векторная прецессия Ландау-Лифшица-Гильберта (Rodrigues-rotation + демпфирование), СГС.

    bias_axis : {'x','y','z','none'} — гибкость по ориентации подмагничивания,
    'none' — режим без подмагничивания (чистая нутация под действием
    только ВЧ-компонент поля).
    """
    def __init__(self, Ms=0.3, gamma=1.0, alpha=0.1, H0_bias=0.0, bias_axis='x'):
        self.Ms, self.gamma, self.alpha = Ms, gamma, alpha
        self.H0_bias = H0_bias
        self.bias_axis = bias_axis

    @staticmethod
    def _rodrigues_rotate(M, Heff, gamma, dt):
        Hnorm = np.linalg.norm(Heff, axis=0)
        Hnorm_safe = np.where(Hnorm > 1e-300, Hnorm, 1.0)
        axis_ = Heff / Hnorm_safe
        theta = gamma * Hnorm * dt
        ct, st = np.cos(theta), np.sin(theta)
        dot = np.sum(axis_*M, axis=0)
        cross = np.array([axis_[1]*M[2]-axis_[2]*M[1],
                           axis_[2]*M[0]-axis_[0]*M[2],
                           axis_[0]*M[1]-axis_[1]*M[0]])
        M_new = M*ct + cross*st + axis_*dot*(1-ct)
        return np.where(Hnorm > 1e-300, M_new, M)

    def _llg_step_core(self, M, Heff, dt):
        M1 = self._rodrigues_rotate(M, Heff, self.gamma, dt)
        cross1 = np.array([M1[1]*Heff[2]-M1[2]*Heff[1],
                            M1[2]*Heff[0]-M1[0]*Heff[2],
                            M1[0]*Heff[1]-M1[1]*Heff[0]])
        damp_term = np.array([M1[1]*cross1[2]-M1[2]*cross1[1],
                               M1[2]*cross1[0]-M1[0]*cross1[2],
                               M1[0]*cross1[1]-M1[1]*cross1[0]])
        M2 = M1 - dt*(self.gamma*self.alpha/self.Ms)*damp_term
        norm2 = np.linalg.norm(M2, axis=0)
        norm2_safe = np.where(norm2 > 1e-300, norm2, 1.0)
        return M2 * (self.Ms/norm2_safe)

    def build_Heff(self, Hy_rf, Hz_rf):
        n = Hy_rf.shape[0]
        Heff = np.zeros((3, n))
        if self.bias_axis == 'x':
            Heff[0] = self.H0_bias; Heff[1] = Hy_rf; Heff[2] = Hz_rf
        elif self.bias_axis == 'y':
            Heff[1] = self.H0_bias + Hy_rf; Heff[2] = Hz_rf
        elif self.bias_axis == 'z':
            Heff[1] = Hy_rf; Heff[2] = self.H0_bias + Hz_rf
        else:
            Heff[1] = Hy_rf; Heff[2] = Hz_rf
        return Heff

    def initial_M(self, n):
        M = np.zeros((3, n))
        idx = {'x': 0, 'y': 1, 'z': 2}.get(self.bias_axis, 2)
        M[idx, :] = self.Ms
        return M

    def step(self, M, Hy_rf, Hz_rf, dt):
        Heff = self.build_Heff(Hy_rf, Hz_rf)
        M_new = self._llg_step_core(M, Heff, dt)
        dMy_dt = (M_new[1] - M[1]) / dt
        dMz_dt = (M_new[2] - M[2]) / dt
        return M_new, dMy_dt, dMz_dt


class LLGJAHybrid:
    """Связка гистерезиса (JA, компонента Hz) и векторной прецессии (LLG, полное M).

    JA обрабатывает Hz (получая на выходе поле после гистерезисной коррекции),
    LLG обрабатывает полную векторную прецессию, используя ПОСЛЕ-JA поле Hz
    (несёт гистерезисные искажения) — поэтому траектория M физически иная,
    чем у чистого LLG.
    """
    def __init__(self, ja, llg):
        self.ja = ja
        self.llg = llg

    def reset_state(self, n_fer):
        self.M_irr = np.zeros(n_fer)
        self.M_field = np.zeros(n_fer)
        self.M = self.llg.initial_M(n_fer)

    def step(self, Hy_old, Hz_old, rhs_extra_z, dt, n_newton=6):
        Hz_after_JA, M_irr_new, M_field_new = self.ja.implicit_step(
            self.M_irr, Hz_old, self.M_field, rhs_extra_z, dt, n_newton=n_newton)
        self.M_irr, self.M_field = M_irr_new, M_field_new
        M_new, dMy_dt, dMz_dt = self.llg.step(self.M, Hy_old, Hz_after_JA, dt)
        self.M = M_new
        return Hz_after_JA, dMy_dt, dMz_dt


class MenDriveFDTD:
    """FDTD-схема резонатора МенДрайв (единицы СГС).

    Гибкость по требованиям:
    - ferrite_model: 'JA' | 'LLG' | 'Hybrid'
    - bias_orientation: 'none'|'x'|'y'|'z' (LLG/Hybrid), 'parallel'|'none' (JA)
    - excitation_mode: 'magnetic_right' (магнитный ток по скин-слою правой
      стенки) | 'electric_left' (электрический ток по скин-слою левого
      проводника)
    - sigma_e_left: варьируемые электрические потери левого проводника
    """
    C_PHYS_M_S = 3e8

    def __init__(self, N, a=0.2, h_l=0.2, h_r=0.2, dt_frac=0.1,
                 excitation_mode='magnetic_right', ferrite_model='JA',
                 bias_orientation='none', H0_bias=0.0,
                 sigma_m_leak=3.0, sigma_e_left=3.0, sigma_e_ferrite=0.0,
                 Ms=1.0, a_JA=0.3, alpha_JA=0.001, k_JA=0.15, c_JA=0.15,
                 n_sub=4, n_fp=3, n_newton=6,
                 Ms_llg=0.3, gamma_llg=1.0, alpha_llg=0.1, skin_depth=0.06):
        self.N, self.a, self.h_l, self.h_r = N, a, h_l, h_r
        self.excitation_mode = excitation_mode
        self.ferrite_model = ferrite_model
        self.n_newton = n_newton
        x_min, x_max = -(a + h_l), (a + h_r)
        self.dx = (x_max - x_min) / N
        self.xA = x_min + self.dx * np.arange(N + 1)
        self.xB = x_min + self.dx * (np.arange(N) + 0.5)
        self.leftA, self.rightA = self.xA < -a, self.xA > a
        self.leftB, self.rightB = self.xB < -a, self.xB > a
        self.iL_A = np.argmin(np.abs(self.xA + a))
        self.iR_A = np.argmin(np.abs(self.xA - a))
        self.dt = dt_frac * self.dx
        self.n_fer = int(self.rightB.sum())
        self.n_left = int(self.leftB.sum())
        self.bias_orientation = bias_orientation

        if ferrite_model == 'JA':
            H0_eff = H0_bias if bias_orientation in ('parallel', 'z') else 0.0
            self.ja = JASmoothRK4Robust(Ms, a_JA, alpha_JA, k_JA, c_JA,
                                         sigma_m_leak=sigma_m_leak, H0_bias=H0_eff,
                                         n_sub=n_sub, n_fp=n_fp)
            self.ferrite_engine = self.ja
        elif ferrite_model == 'LLG':
            axis = bias_orientation if bias_orientation in ('x', 'y', 'z') else 'none'
            self.llg = LLGVector(Ms=Ms_llg, gamma=gamma_llg, alpha=alpha_llg,
                                  H0_bias=H0_bias, bias_axis=axis)
            self.ferrite_engine = self.llg
        else:
            self.ja = JASmoothRK4Robust(Ms, a_JA, alpha_JA, k_JA, c_JA,
                                         sigma_m_leak=sigma_m_leak, H0_bias=0.0,
                                         n_sub=n_sub, n_fp=n_fp)
            axis = bias_orientation if bias_orientation in ('x', 'y', 'z') else 'none'
            self.llg = LLGVector(Ms=Ms_llg, gamma=gamma_llg, alpha=alpha_llg,
                                  H0_bias=H0_bias, bias_axis=axis)
            self.hybrid = LLGJAHybrid(self.ja, self.llg)
            self.ferrite_engine = self.hybrid

        sigma_e_A = np.where(self.leftA, sigma_e_left, np.where(self.rightA, sigma_e_ferrite, 0.0))
        self.Ca_e = (1 - 2*np.pi*sigma_e_A*self.dt) / (1 + 2*np.pi*sigma_e_A*self.dt)
        self.Cb_e = (self.dt/self.dx) / (1 + 2*np.pi*sigma_e_A*self.dt)
        self.src_mag_B = np.where(self.rightB, np.exp(-(self.xB - a)/skin_depth), 0.0)
        self.src_el_A = np.where(self.leftA, np.exp(-(-a - self.xA)/skin_depth), 0.0)
        self.reset_state()

    def reset_state(self):
        N = self.N
        self.Ey, self.Ez = np.zeros(N+1), np.zeros(N+1)
        self.Hy, self.Hz = np.zeros(N), np.zeros(N)
        self.t = 0.0
        self._last_HzJA = None
        if self.ferrite_model == 'JA':
            self.M_irr = np.zeros(self.n_fer)
            self.M_field = np.zeros(self.n_fer)
        elif self.ferrite_model == 'LLG':
            self.M = self.llg.initial_M(self.n_fer)
        else:
            self.hybrid.reset_state(self.n_fer)
            self._last_HzJA = np.zeros(self.n_fer)

    def _step(self, omega0, amp, ramp_periods, T):
        dt, dx = self.dt, self.dx
        ramp = min(1.0, self.t / (ramp_periods * T))
        s = amp * ramp * np.sin(omega0 * self.t)
        j_e_y_A = np.zeros(self.N + 1)
        j_m_z_B = np.zeros(self.N)
        if self.excitation_mode == 'magnetic_right':
            j_m_z_B = s * self.src_mag_B
        elif self.excitation_mode == 'electric_left':
            j_e_y_A = s * self.src_el_A

        Ey, Ez, Hy, Hz = self.Ey, self.Ez, self.Hy, self.Hz
        Ey_new = np.zeros_like(Ey)
        Ez_new = np.zeros_like(Ez)
        Ey_new[1:-1] = (self.Ca_e[1:-1]*Ey[1:-1] - self.Cb_e[1:-1]*(Hz[1:]-Hz[:-1])
                        - dt*4*np.pi*j_e_y_A[1:-1])
        Ez_new[1:-1] = self.Ca_e[1:-1]*Ez[1:-1] + self.Cb_e[1:-1]*(Hy[1:]-Hy[:-1])
        Ey_new[0] = Ey_new[-1] = 0.0
        Ez_new[0] = Ez_new[-1] = 0.0
        P_e = -np.sum(j_e_y_A * 0.5*(Ey + Ey_new)) * dx
        Ey, Ez = Ey_new, Ez_new

        rotE_z = (Ey[1:] - Ey[:-1]) / dx
        rightB = self.rightB
        Hz_new = np.zeros_like(Hz)
        Hy_new = Hy + (dt/dx)*(Ez[1:]-Ez[:-1])
        Hz_new[~rightB] = Hz[~rightB] - dt*rotE_z[~rightB] - dt*4*np.pi*j_m_z_B[~rightB]

        if self.ferrite_model == 'JA':
            rhs_extra = dt*rotE_z[rightB] + dt*4*np.pi*j_m_z_B[rightB]
            Hz_f, M_irr_new, M_new = self.ja.implicit_step(self.M_irr, Hz[rightB], self.M_field,
                                                             rhs_extra, dt, n_newton=self.n_newton)
            Hz_new[rightB] = Hz_f
            self.M_irr, self.M_field = M_irr_new, M_new
        elif self.ferrite_model == 'LLG':
            Hy_rf = Hy[rightB]
            Hz_rf = Hz[rightB]
            M_new, dMy_dt, dMz_dt = self.llg.step(self.M, Hy_rf, Hz_rf, dt)
            self.M = M_new
            Hy_new[rightB] = Hy[rightB] + (dt/dx)*(Ez[1:]-Ez[:-1])[rightB] - dt*4*np.pi*dMy_dt
            Hz_new[rightB] = (Hz[rightB] - dt*rotE_z[rightB] - dt*4*np.pi*j_m_z_B[rightB]
                              - dt*4*np.pi*dMz_dt)
        else:
            rhs_extra_z = dt*rotE_z[rightB] + dt*4*np.pi*j_m_z_B[rightB]
            Hy_rf = Hy[rightB]
            Hz_after_JA, dMy_dt, dMz_dt = self.hybrid.step(Hy_rf, Hz[rightB], rhs_extra_z, dt,
                                                            n_newton=self.n_newton)
            self._last_HzJA = Hz_after_JA
            Hy_new[rightB] = Hy[rightB] + (dt/dx)*(Ez[1:]-Ez[:-1])[rightB] - dt*4*np.pi*dMy_dt
            Hz_new[rightB] = Hz_after_JA - dt*4*np.pi*dMz_dt

        P_m = -np.sum(j_m_z_B * 0.5*(Hz + Hz_new)) * dx
        self.Ey, self.Ez, self.Hy, self.Hz = Ey, Ez, Hy_new, Hz_new
        self.t += dt
        return P_e + P_m

    def Txx_at(self, iA):
        Hy_A = 0.5*(self.Hy[iA-1]+self.Hy[iA]) if 0 < iA < self.N else 0.0
        Hz_A = 0.5*(self.Hz[iA-1]+self.Hz[iA]) if 0 < iA < self.N else 0.0
        return (self.Ey[iA]**2 + self.Ez[iA]**2 + Hy_A**2 + Hz_A**2) / (8*np.pi)

    def run(self, omega0, n_periods, record_from_period, amp=1.0, ramp_periods=2,
            probe_idx=0, time_budget=None):
        """Прогон схемы с записью dTxx (интеграл тензора Максвелла), Hz-отклика,
        компонент M, внутреннего поля/намагниченности JA (для Hybrid) и
        мощности источника P.
        """
        T = 2*np.pi/omega0
        nsteps = int(n_periods*T/self.dt)
        record_start = int(record_from_period*T/self.dt)
        rec_t, rec_dTxx, rec_Hn, rec_Mn, rec_HzJA, rec_MzJA, rec_P = [], [], [], [], [], [], []
        blew_up = False
        t0 = time.time()
        for n in range(nsteps):
            P = self._step(omega0, amp, ramp_periods, T)
            if n >= record_start:
                rec_t.append(self.t)
                rec_dTxx.append(self.Txx_at(self.iR_A) - self.Txx_at(self.iL_A))
                rec_Hn.append(self.Hz[self.rightB][probe_idx])
                if self.ferrite_model == 'JA':
                    rec_Mn.append(self.M_field[probe_idx])
                elif self.ferrite_model == 'LLG':
                    rec_Mn.append(self.M[:, probe_idx].copy())
                else:
                    rec_Mn.append(self.hybrid.M[:, probe_idx].copy())
                    rec_HzJA.append(self._last_HzJA[probe_idx])
                    rec_MzJA.append(self.hybrid.M_field[probe_idx])
                rec_P.append(P/self.dt)
            if not np.isfinite(self.Hz).all():
                blew_up = True
                break
            if time_budget is not None and time.time() - t0 > time_budget:
                break
        rec_Mn = np.array(rec_Mn)
        return dict(t=np.array(rec_t), dTxx=np.array(rec_dTxx), Hn=np.array(rec_Hn),
                    Mn=rec_Mn, HzJA=np.array(rec_HzJA), MzJA=np.array(rec_MzJA),
                    P=np.array(rec_P), T=T, dt=self.dt, blew_up=blew_up)

    def force_per_power(self, res, last_frac=0.5):
        """Возвращает (отношение в кодовых единицах, силу на киловатт в Н/кВт)."""
        T, dt = res['T'], res['dt']
        spp = int(round(T/dt))
        n_rec = max(1, len(res['t']) // spp)
        pm_F = np.array([res['dTxx'][i*spp:(i+1)*spp].mean() for i in range(n_rec)])
        pm_P = np.array([res['P'][i*spp:(i+1)*spp].mean() for i in range(n_rec)])
        k = max(1, int(len(pm_F)*last_frac))
        mean_F, mean_P = pm_F[-k:].mean(), pm_P[-k:].mean()
        ratio_code = mean_F/mean_P if mean_P != 0 else np.nan
        return ratio_code, ratio_code/self.C_PHYS_M_S*1000


def shoelace_area(x, y):
    """Площадь замкнутой кривой по формуле шнуровки.

    Используется как объективная проверка петли гистерезиса: площадь > 0
    означает реальную замкнутую петлю, площадь ~ 0 означает вырождение
    в линию (что было бы признаком поломанной модели).
    """
    return 0.5*abs(np.sum(x*np.roll(y, -1) - np.roll(x, -1)*y))


def scan_resonance(N, freqs, ferrite_model='JA', time_budget=3.0,
                    n_sub=1, n_fp=1, n_newton=3, **kwargs):
    """Скан амплитуды отклика Hz по сетке частот.

    КРИТИЧЕСКИЙ ФИКС: дешёвые настройки Ньютона (n_sub, n_fp, n_newton)
    передаются во ВСЕ три движка (JA, LLG, Hybrid), а не только в JA.
    Иначе Hybrid почти всегда не успевает дойти до начала записи в
    пределах time_budget (JA-шаг внутри Hybrid в ~340 раз дороже LLG-шага),
    что даёт скан из одной "точки" вместо полного спектра отклика.
    """
    resp = []
    for w in freqs:
        sim = MenDriveFDTD(N, ferrite_model=ferrite_model,
                           n_sub=n_sub, n_fp=n_fp, n_newton=n_newton, **kwargs)
        res = sim.run(w, n_periods=3, record_from_period=1, amp=0.3, ramp_periods=1,
                      time_budget=time_budget)
        a_resp = (res['Hn'].max()-res['Hn'].min())/2 if (len(res['Hn']) > 0 and not res['blew_up']) else np.nan
        resp.append(a_resp)
    return np.array(resp)


def main(N=20, ferrite_model='Hybrid', bias_orientation='x', H0_bias=4.0,
         excitation_mode='magnetic_right', sigma_e_left=3.0, sigma_m_leak=3.0,
         Ms=1.0, a_JA=0.3, alpha_JA=0.001, k_JA=0.15, c_JA=0.15,
         Ms_llg=0.3, gamma_llg=1.0, alpha_llg=0.1,
         n_sub=2, n_fp=2, n_newton=4,
         freqs_wide=None, freqs_fine_halfwidth=0.9, freqs_fine_step=0.1,
         scan_time_budget=5.0,
         n_periods_total=80, record_from_period=15, amp=1.0, ramp_periods=2.0,
         run_time_budget=350.0, probe_idx=0, last_frac_force=0.5,
         make_plots=True, out_dir='./outputs', verbose=True):
    """
    Полный цикл анализа резонатора МенДрайв.

    Этапы:
      1) Грубый скан резонанса по широкой сетке частот, затем уточняющий
         скан узкой сеткой вокруг найденного грубого пика.
      2) Длинный прогон FDTD на уточнённой резонансной частоте с записью
         динамики тензора Максвелла (dTxx), поля Hz, компонент M и мощности.
      3) FFT-анализ динамики dTxx (диагностика биений/квазипериодичности).
      4) Проверка сходимости накопленного среднего отношения тяга/мощность
         по числу усреднённых периодов записи.
      5) Петля гистерезиса: для 'JA' — прямая петля H-B; для 'Hybrid' —
         внутренняя (неискажённая последующей LLG-коррекцией) петля JA-компонента.
      6) Для 'Hybrid' дополнительно запускается опорный прогон чистого LLG
         с теми же параметрами подмагничивания — для сравнения траекторий
         намагниченности (диагностика искажения орбиты M гистерезисом).

    Параметры
    ---------
    N : int
        Число ячеек сетки FDTD.
    ferrite_model : {'JA', 'LLG', 'Hybrid'}
        Модель феррита: чистый гистерезис Джилса-Атертона, чистая векторная
        прецессия Ландау-Лифшица-Гильберта, либо их связка.
    bias_orientation : {'none', 'x', 'y', 'z', 'parallel'}
        Ориентация подмагничивающего поля. 'parallel'/'z' действуют как
        продольное поле для JA; 'x'/'y'/'z' — оси подмагничивания для
        LLG/Hybrid; 'none' — режим без подмагничивания.
    H0_bias : float
        Величина подмагничивающего поля H0 (Э, СГС).
    excitation_mode : {'magnetic_right', 'electric_left'}
        Способ возбуждения: магнитный ток по скин-слою правой стенки
        (гибридный резонатор с ферритом) либо электрический ток по
        скин-слою левого проводника.
    sigma_e_left : float
        Электрическая проводимость левого проводника — варьируемые
        электрические потери.
    sigma_m_leak : float
        Магнитные потери (утечка) в феррите (для JA/Hybrid).
    Ms, a_JA, alpha_JA, k_JA, c_JA : float
        Параметры модели Джилса-Атертона (насыщение, форм-параметр,
        межзёренное взаимодействие, коэрцитивность, обратимая доля).
    Ms_llg, gamma_llg, alpha_llg : float
        Параметры LLG (намагниченность насыщения, гиромагнитное отношение,
        параметр демпфирования Гильберта).
    n_sub, n_fp, n_newton : int
        "Дорогие" настройки точности JA-решателя для финального длинного
        прогона (в сканах резонанса всегда используются отдельные дешёвые
        настройки n_sub=1, n_fp=1, n_newton=3 — см. scan_resonance).
    freqs_wide : array или None
        Сетка частот для грубого скана. По умолчанию np.arange(1.0, 16.01, 0.5).
    freqs_fine_halfwidth, freqs_fine_step : float
        Полуширина и шаг уточняющего скана вокруг грубого пика.
    scan_time_budget : float
        Бюджет времени (с) на одну точку скана.
    n_periods_total, record_from_period : int
        Общая длительность финального прогона (в периодах возбуждения) и
        номер периода, с которого начинается запись.
    amp, ramp_periods : float
        Амплитуда возбуждения и число периодов линейного нарастания (ramp).
    run_time_budget : float
        Общий бюджет времени (с) на финальный длинный прогон.
    probe_idx : int
        Индекс ячейки внутри феррита, откуда снимаются локальные величины
        (Hz, M, HzJA, MzJA) для петли гистерезиса и траектории M.
    last_frac_force : float
        Доля последних периодов записи, усредняемая для force_per_power.
    make_plots : bool
        Сохранять ли PNG-графики в out_dir.
    out_dir : str
        Каталог для сохранения графиков.
    verbose : bool
        Печатать ли ход выполнения и промежуточные метрики.

    Возвращает
    ----------
    dict с ключами:
        omega_res              -- уточнённая резонансная частота
        freqs_wide, resp_wide  -- сетка и отклик грубого скана
        freqs_fine, resp_fine  -- сетка и отклик уточняющего скана
        res_long                -- полный словарь результатов длинного прогона
        res_long_llg_ref         -- аналогичный прогон чистого LLG (только
                                    если ferrite_model == 'Hybrid', иначе None)
        hysteresis_area          -- площадь петли гистерезиса (JA или
                                    внутренний JA-контур Hybrid), None для LLG
        fft_freqs, fft_mag       -- спектр |FFT(dTxx)|
        convergence_ratio_cum    -- накопленное среднее force/power по
                                    числу усреднённых периодов записи
        force_per_power_code     -- итоговое отношение тяга/мощность (код.ед.)
        force_per_kW             -- то же в физических единицах Н/кВт
        plots                    -- список путей сохранённых PNG-файлов
    """
    os.makedirs(out_dir, exist_ok=True)
    t_start = time.time()
    common_kwargs = dict(bias_orientation=bias_orientation, H0_bias=H0_bias,
                          Ms_llg=Ms_llg, gamma_llg=gamma_llg, alpha_llg=alpha_llg,
                          Ms=Ms, a_JA=a_JA, alpha_JA=alpha_JA, k_JA=k_JA, c_JA=c_JA,
                          sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                          excitation_mode=excitation_mode)
    plots = []

    # ---------- (1a) Грубый скан резонанса ----------
    if freqs_wide is None:
        freqs_wide = np.arange(1.0, 16.01, 0.5)
    resp_wide = scan_resonance(N, freqs_wide, ferrite_model=ferrite_model,
                                time_budget=scan_time_budget,
                                n_sub=1, n_fp=1, n_newton=3, **common_kwargs)
    if np.all(np.isnan(resp_wide)):
        raise RuntimeError("Грубый скан не дал ни одной валидной точки -- "
                            "увеличьте scan_time_budget или уменьшите N.")
    omega_coarse = freqs_wide[np.nanargmax(resp_wide)]
    if verbose:
        print(f"[main] Грубый скан: пик omega0~{omega_coarse:.2f} "
              f"({np.sum(~np.isnan(resp_wide))}/{len(freqs_wide)} валидных точек), "
              f"{time.time()-t_start:.1f}с", flush=True)

    # ---------- (1b) Уточняющий скан вокруг грубого пика ----------
    lo = max(freqs_wide[0], omega_coarse - freqs_fine_halfwidth)
    hi = omega_coarse + freqs_fine_halfwidth
    freqs_fine = np.arange(lo, hi + 1e-9, freqs_fine_step)
    resp_fine = scan_resonance(N, freqs_fine, ferrite_model=ferrite_model,
                                time_budget=scan_time_budget*1.5,
                                n_sub=1, n_fp=1, n_newton=3, **common_kwargs)
    if np.all(np.isnan(resp_fine)):
        omega_res = omega_coarse
        if verbose:
            print("[main] Уточняющий скан не дал валидных точек, использую грубый пик.")
    else:
        omega_res = freqs_fine[np.nanargmax(resp_fine)]
    if verbose:
        print(f"[main] Уточнённый резонанс: omega0={omega_res:.3f}, "
              f"{time.time()-t_start:.1f}с", flush=True)

    if make_plots:
        plt.figure(figsize=(8, 4))
        plt.plot(freqs_wide, resp_wide, 'o--', ms=3, alpha=0.5, label='грубый скан')
        plt.plot(freqs_fine, resp_fine, 'o-', ms=4, color='tab:blue', label='уточняющий скан')
        plt.axvline(omega_res, color='red', ls='--', label=f'omega0={omega_res:.2f}')
        plt.xlabel('omega0'); plt.ylabel('амплитуда отклика Hz')
        plt.title(f'{ferrite_model}: скан резонанса')
        plt.legend(); plt.grid(True); plt.tight_layout()
        p = os.path.join(out_dir, f'scan_{ferrite_model}.png')
        plt.savefig(p, dpi=120); plt.close()
        plots.append(p)

    # ---------- (2) Длинный прогон на найденном резонансе ----------
    sim = MenDriveFDTD(N, ferrite_model=ferrite_model, n_sub=n_sub, n_fp=n_fp,
                        n_newton=n_newton, **common_kwargs)
    res_long = sim.run(omega_res, n_periods=n_periods_total,
                        record_from_period=record_from_period, amp=amp,
                        ramp_periods=ramp_periods, probe_idx=probe_idx,
                        time_budget=run_time_budget)
    n_periods_cov = (len(res_long['t'])*res_long['dt']/res_long['T']
                      if len(res_long['t']) > 0 else 0.0)
    if verbose:
        print(f"[main] Длинный прогон ({ferrite_model}): {time.time()-t_start:.1f}с, "
              f"точек={len(res_long['t'])}, blew_up={res_long['blew_up']}, "
              f"покрыто периодов записи~{n_periods_cov:.1f}", flush=True)

    # ---------- Опциональный опорный прогон чистого LLG (для сравнения траекторий) ----------
    res_llg_ref = None
    if ferrite_model == 'Hybrid':
        llg_kwargs = dict(bias_orientation=bias_orientation, H0_bias=H0_bias,
                           Ms_llg=Ms_llg, gamma_llg=gamma_llg, alpha_llg=alpha_llg,
                           sigma_e_left=sigma_e_left, excitation_mode=excitation_mode)
        sim_llg = MenDriveFDTD(N, ferrite_model='LLG', **llg_kwargs)
        res_llg_ref = sim_llg.run(omega_res, n_periods=n_periods_total,
                                   record_from_period=record_from_period, amp=amp,
                                   ramp_periods=ramp_periods, probe_idx=probe_idx,
                                   time_budget=max(60.0, run_time_budget*0.2))
        if verbose:
            print(f"[main] Опорный прогон чистого LLG: точек={len(res_llg_ref['t'])}, "
                  f"blew_up={res_llg_ref['blew_up']}", flush=True)

    # ---------- (3) FFT динамики тензора ----------
    t_h, dTxx_h, dt_h, T_h = res_long['t'], res_long['dTxx'], res_long['dt'], res_long['T']
    fft_freqs, fft_mag = np.array([]), np.array([])
    if len(dTxx_h) >= 8:
        Nfft = len(dTxx_h)
        window = np.hanning(Nfft)
        spec = np.fft.rfft(dTxx_h * window)
        fft_freqs = np.fft.rfftfreq(Nfft, d=dt_h) * 2*np.pi
        fft_mag = np.abs(spec)
        if make_plots:
            plt.figure(figsize=(9, 4))
            plt.plot(fft_freqs, fft_mag, lw=1.0)
            plt.axvline(omega_res, color='red', ls='--', alpha=0.6, label=f'omega0={omega_res:.2f}')
            if bias_orientation in ('x', 'y', 'z') and ferrite_model in ('LLG', 'Hybrid'):
                omega_larmor = gamma_llg * H0_bias
                plt.axvline(omega_larmor, color='green', ls='--', alpha=0.6,
                            label=f'gamma*H0={omega_larmor:.2f}')
            xmax = min(fft_freqs.max() if len(fft_freqs) else 20, 4*omega_res)
            plt.xlim(0, xmax)
            plt.xlabel('omega (рад/ед.времени)'); plt.ylabel('|FFT(dTxx)|')
            plt.title(f'{ferrite_model}: спектр динамики тензора dTxx')
            plt.legend(); plt.grid(True); plt.tight_layout()
            p = os.path.join(out_dir, f'fft_dTxx_{ferrite_model}.png')
            plt.savefig(p, dpi=120); plt.close()
            plots.append(p)
    elif verbose:
        print("[main] Слишком мало точек для FFT -- пропускаю спектральный анализ.")

    # ---------- (4) Проверка сходимости force/power ----------
    convergence_ratio_cum = np.array([])
    if len(t_h) >= 1:
        spp = int(round(T_h/dt_h))
        n_rec_periods = len(t_h) // spp
        if n_rec_periods >= 1:
            pm_dTxx = np.array([dTxx_h[i*spp:(i+1)*spp].mean() for i in range(n_rec_periods)])
            pm_P = np.array([res_long['P'][i*spp:(i+1)*spp].mean() for i in range(n_rec_periods)])
            cum_mean_dTxx = np.cumsum(pm_dTxx) / np.arange(1, n_rec_periods+1)
            cum_mean_P = np.cumsum(pm_P) / np.arange(1, n_rec_periods+1)
            convergence_ratio_cum = np.where(np.abs(cum_mean_P) > 1e-30,
                                              cum_mean_dTxx/cum_mean_P, np.nan)
            if make_plots:
                plt.figure(figsize=(9, 4))
                plt.plot(np.arange(1, n_rec_periods+1), convergence_ratio_cum, 'o-', ms=3)
                plt.xlabel('число усреднённых периодов записи')
                plt.ylabel('накопленное среднее dTxx/P')
                plt.title(f'{ferrite_model}: сходимость отношения тяга/мощность')
                plt.grid(True); plt.tight_layout()
                p = os.path.join(out_dir, f'convergence_{ferrite_model}.png')
                plt.savefig(p, dpi=120); plt.close()
                plots.append(p)
            if verbose:
                for frac in [0.25, 0.5, 0.75, 1.0]:
                    i = max(0, int(n_rec_periods*frac)-1)
                    print(f"[main] Сходимость {int(frac*100):3d}% записи: "
                          f"ratio={convergence_ratio_cum[i]:.4e}")

    # ---------- (5) Петля гистерезиса (JA внутри Hybrid, либо чистая JA) ----------
    hysteresis_area = None
    spp_h = int(round(T_h/dt_h)) if len(t_h) > 0 else 0
    if ferrite_model == 'Hybrid' and spp_h > 0 and len(res_long['HzJA']) >= spp_h:
        Hloop = res_long['HzJA'][-spp_h:]
        Bloop = Hloop + 4*np.pi*res_long['MzJA'][-spp_h:]
        hysteresis_area = shoelace_area(Hloop, Bloop)
        if make_plots:
            plt.figure(figsize=(5, 5))
            plt.plot(Hloop, Bloop, lw=1.5, color='tab:green', marker='o', ms=2)
            plt.xlabel('H (Э), поле, видимое JA-компонентом'); plt.ylabel('B (Гс)')
            plt.title(f'Внутренняя петля JA внутри Hybrid (площадь={hysteresis_area:.3e})')
            plt.grid(True); plt.tight_layout()
            p = os.path.join(out_dir, 'hysteresis_Hybrid_internalJA.png')
            plt.savefig(p, dpi=120); plt.close()
            plots.append(p)
    elif ferrite_model == 'JA' and spp_h > 0 and len(res_long['Hn']) >= spp_h:
        Hloop = res_long['Hn'][-spp_h:]
        Bloop = Hloop + 4*np.pi*res_long['Mn'][-spp_h:]
        hysteresis_area = shoelace_area(Hloop, Bloop)
        if make_plots:
            plt.figure(figsize=(5, 5))
            plt.plot(Hloop, Bloop, lw=1.5, color='tab:purple', marker='o', ms=2)
            plt.xlabel('H (Э)'); plt.ylabel('B (Гс)')
            plt.title(f'Петля гистерезиса JA (площадь={hysteresis_area:.3e})')
            plt.grid(True); plt.tight_layout()
            p = os.path.join(out_dir, 'hysteresis_JA.png')
            plt.savefig(p, dpi=120); plt.close()
            plots.append(p)

    # ---------- (6) Сравнение траектории M: Hybrid vs чистый LLG ----------
    if ferrite_model == 'Hybrid' and res_llg_ref is not None and spp_h > 0:
        n_traj = min(3*spp_h, len(res_long['Mn']), len(res_llg_ref['Mn']))
        if n_traj > 0:
            fig, axes = plt.subplots(1, 2, figsize=(11, 5))
            axes[0].plot(res_long['Mn'][-n_traj:, 1], res_long['Mn'][-n_traj:, 2],
                         lw=0.8, color='tab:green')
            axes[0].set_xlabel('M_y'); axes[0].set_ylabel('M_z')
            axes[0].set_title('Hybrid: траектория M')
            axes[0].set_aspect('equal', adjustable='box'); axes[0].grid(True)
            axes[1].plot(res_llg_ref['Mn'][-n_traj:, 1], res_llg_ref['Mn'][-n_traj:, 2],
                         lw=0.8, color='tab:orange')
            axes[1].set_xlabel('M_y'); axes[1].set_ylabel('M_z')
            axes[1].set_title('Чистый LLG: траектория M')
            axes[1].set_aspect('equal', adjustable='box'); axes[1].grid(True)
            plt.suptitle(f'Сравнение траекторий намагниченности, omega0={omega_res:.2f}')
            plt.tight_layout()
            p = os.path.join(out_dir, 'M_trajectory_Hybrid_vs_LLG.png')
            plt.savefig(p, dpi=120); plt.close()
            plots.append(p)

    # ---------- Итоговая оценка тяги на единицу мощности ----------
    force_per_power_code, force_per_kW = sim.force_per_power(res_long, last_frac=last_frac_force)
    if verbose:
        print(f"[main] FORCE/POWER (последние {int(last_frac_force*100)}% записи): "
              f"{force_per_power_code:.4e} (код.ед.), {force_per_kW:.4e} Н/кВт")
        print(f"[main] Итого времени: {time.time()-t_start:.1f}с")

    return dict(omega_res=omega_res, freqs_wide=freqs_wide, resp_wide=resp_wide,
                freqs_fine=freqs_fine, resp_fine=resp_fine,
                res_long=res_long, res_long_llg_ref=res_llg_ref,
                hysteresis_area=hysteresis_area,
                fft_freqs=fft_freqs, fft_mag=fft_mag,
                convergence_ratio_cum=convergence_ratio_cum,
                force_per_power_code=force_per_power_code, force_per_kW=force_per_kW,
                plots=plots)


if __name__ == "__main__":
    # Пример запуска для боевых (не тестовых) настроек точности:
    result = main(N=80, ferrite_model='Hybrid', bias_orientation='x', H0_bias=4.0,
                  n_sub=2, n_fp=2, n_newton=4,
                  n_periods_total=80, record_from_period=15,
                  run_time_budget=None)
    print("omega_res =", result['omega_res'])
    print("force_per_kW =", result['force_per_kW'])
    print("hysteresis_area =", result['hysteresis_area'])